In [ ]:
import csv
import json
import numpy as np
import pandas as pd
import requests

# ---------------------------------------------------------
# Task 1.1 — Read with csv module & validate numeric types
# ---------------------------------------------------------
with open("appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    raw_rows = list(reader)

print("Header:", header)
print("Row count:", len(raw_rows))


def parse_row(row):
    parsed = dict(row)
    parsed["age"] = int(parsed["age"])
    parsed["fee_inr"] = int(parsed["fee_inr"])
    parsed["prior_no_shows"] = int(parsed["prior_no_shows"])
    parsed["distance_km"] = float(parsed["distance_km"])
    return parsed


accepted = []
rejected = []
for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError as err:
        rejected.append({"row": row, "error": str(err)})

print(f"Accepted: {len(accepted)}, Rejected: {len(rejected)}")
with open("rejected_rows.json", "w", encoding="utf-8") as jf:
    json.dump(rejected, jf, indent=2)

# ---------------------------------------------------------
# Task 1.2 — Inspect and clean, pass 1 (fresh read)
# ---------------------------------------------------------
df = pd.read_csv("appointments.csv")
print("\nShape:", df.shape)
print(df.info())
print("\nNull counts:\n", df.isnull().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Raw booking_channel unique count:", df["booking_channel"].nunique(dropna=False))

df = df.drop_duplicates()
print("Row count after (a) dropping duplicates:", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Cleaned booking_channel values:", sorted(df["booking_channel"].dropna().unique()))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
df = df.dropna(subset=["appt_date", "status"])
print("Row count after (c)-(d) dropping missing appt_date or status:", len(df))

# ---------------------------------------------------------
# Task 1.3 — Clean, pass 2: validity, gaps and outliers
# ---------------------------------------------------------
invalid_age = (df["age"] < 1) | (df["age"] > 100)
df.loc[invalid_age, "age"] = np.nan
print("\nTotal missing age after invalid check:", int(df["age"].isnull().sum()))

med_age = df["age"].median()
med_dist = df["distance_km"].median()
print(f"Median age: {med_age}, Median distance_km: {med_dist}")

df["age"] = df["age"].fillna(med_age).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(med_dist)

for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_capped = int(((df[col] < lower) | (df[col] > upper)).sum())
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col}: Q1={q1}, Q3={q3}, lower={lower}, upper={upper}, capped={n_capped}")

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Max fee_inr after capping:", int(df["fee_inr"].max()))

# ---------------------------------------------------------
# Task 1.4 — Features and NumPy summary
# ---------------------------------------------------------
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
col_means = X.mean(axis=0)
print("\nColumn means [age, distance_km, fee_inr, lead_days]:", col_means)

lead_mask = df["lead_days"].to_numpy() > 14
print(f"lead_days > 14: count={int(lead_mask.sum())}, share={lead_mask.mean() * 100:.1f}%")

dist_x = df["distance_km"].to_numpy()
z_scores = (dist_x - np.mean(dist_x)) / np.std(dist_x)
print("Rows with |z| > 2 for distance_km:", int((np.abs(z_scores) > 2).sum()))

# ---------------------------------------------------------
# Task 1.5 — Enrich clinics from Open-Meteo API
# ---------------------------------------------------------
clinics_df = pd.read_csv("clinics.csv")
temps = []

for _, c_row in clinics_df.iterrows():
    temp_val = None
    reason = "unknown error"
    for _ in range(2):  # initial attempt + 1 retry
        try:
            resp = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": c_row["latitude"],
                    "longitude": c_row["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1,
                },
                timeout=10,
            )
            if resp.status_code == 200:
                payload = resp.json()
                t_series = payload.get("hourly", {}).get("temperature_2m", [])
                if len(t_series) > 0 and t_series[0] is not None:
                    temp_val = float(t_series[0])
                    break
                reason = "missing hourly.temperature_2m values"
            else:
                reason = f"status_code={resp.status_code}"
        except requests.exceptions.RequestException as req_err:
            reason = str(req_err)
    if temp_val is None:
        print(f"API failed for {c_row['clinic_name']} ({c_row['clinic_id']}): {reason}")
    temps.append(temp_val)

clinics_enriched = clinics_df.copy()
clinics_enriched["temp_c"] = temps
clinics_enriched.to_csv("clinics_enriched.csv", index=False)

# ---------------------------------------------------------
# Task 1.6 — Export clean CSV and write load_data.sql
# ---------------------------------------------------------
df_export = df.sort_values("appointment_id").copy()
df_export["booked_on"] = df_export["booked_on"].dt.strftime("%Y-%m-%d")
df_export["appt_date"] = df_export["appt_date"].dt.strftime("%Y-%m-%d")

final_cols = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show",
]
df_export = df_export[final_cols]

assert df_export["appointment_id"].nunique() == len(df_export), "Duplicate appointment_id!"
assert int(df_export.isnull().sum().sum()) == 0, "Missing values found!"
print(f"\nUnique appointment_id: {df_export['appointment_id'].nunique()}, Missing values: 0")

df_export.to_csv("appointments_clean.csv", index=False)

sql_statements = []
for _, r in clinics_enriched.iterrows():
    temp_sql = "NULL" if pd.isna(r["temp_c"]) else f"{r['temp_c']}"
    sql_statements.append(
        f"INSERT INTO clinics VALUES ('{r['clinic_id']}','{r['clinic_name']}','{r['city']}','{r['tier']}',{r['latitude']},{r['longitude']},{temp_sql});"
    )

for _, r in df_export.iterrows():
    sql_statements.append(
        f"INSERT INTO appointments VALUES ('{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}','{r['booked_on']}','{r['appt_date']}',{int(r['age'])},'{r['gender']}',{r['distance_km']},{int(r['fee_inr'])},'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',{int(r['prior_no_shows'])},'{r['status']}',{int(r['lead_days'])},{int(r['no_show'])});"
    )

with open("load_data.sql", "w", encoding="utf-8") as sql_file:
    sql_file.write("\n".join(sql_statements) + "\n")

print("Total INSERT statements written to load_data.sql:", len(sql_statements))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020
Accepted: 988, Rejected: 32

Shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-